# DeepAgents Advanced Features

This notebook covers five core capabilities that make DeepAgents powerful for production agent work:

1. **Filesystem** — read/write/edit files in a virtual workspace
2. **Memory (AGENTS.md)** — persistent preferences and context across sessions
3. **Skills (SKILL.md)** — specialized, on-demand instructions
4. **Subagents** — delegate subtasks to specialized child agents
5. **Streaming** — get responses token by token as the agent works

## Setup

In [19]:
import warnings
warnings.filterwarnings('ignore', category=PendingDeprecationWarning)

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from deepagents import create_deep_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o", temperature=0.7)
print("✓ Setup complete")

✓ Setup complete


---

## Feature 1: Filesystem

The **filesystem** is the backbone of DeepAgents. It provides the agent with tools to:
- Read files
- Write and edit files
- List and find files
- Organize work in a virtual workspace

Built-in tools available to all agents:

| Tool | Description |
|---|---|
| `read_file` | Read a file's content |
| `write_file` | Create or overwrite a file |
| `edit_file` | Apply targeted edits (find/replace) |
| `list_dir` | List directory contents |
| `find_files` | Search for files by pattern |

In [ ]:
import os
from langchain_core.tools import tool

print("=" * 70)
print("FEATURE 1: FILESYSTEM - Agent Using write_file Tool")
print("=" * 70)

# Define write_file as an explicit custom tool
@tool
def write_file(filename: str, content: str) -> str:
    """Write content to a file. Returns success message."""
    try:
        with open(filename, "w") as f:
            f.write(content)
        return f"✅ File '{filename}' created successfully ({len(content)} bytes)"
    except Exception as e:
        return f"❌ Error: {str(e)}"

# Create agent with explicit write_file tool
agent = create_deep_agent(
    model=llm,
    tools=[write_file],  # Explicitly provide write_file tool
    system_prompt="""You are a file manager. When asked to create a file, use the write_file tool.""",
)

meeting_content = """# Meeting Notes

## Date
June 27, 2026

## Attendees
- Alice Johnson
- Bob Smith
- Carol Davis

## Topics Discussed
1. Q3 Project Planning
2. Team Capacity Review
3. Risk Assessment

## Action Items
- [ ] Finalize requirements - Alice (Due: June 30)
- [ ] Setup environments - Bob (Due: June 29)
- [ ] Create test plan - Carol (Due: July 1)

## Notes
Productive discussion. Team capacity at 85%.

## Next Meeting
July 1, 2026 at 2:00 PM"""

print("\nInvoking agent to create meeting_notes.md...\n")

# Agent invokes write_file tool
result = agent.invoke({
    "messages": """Create a file called 'meeting_notes.md' with this meeting notes content:

# Meeting Notes

## Date
June 27, 2026

## Attendees
- Alice Johnson
- Bob Smith
- Carol Davis

## Topics Discussed
1. Q3 Project Planning
2. Team Capacity Review
3. Risk Assessment

## Action Items
- [ ] Finalize requirements - Alice (Due: June 30)
- [ ] Setup environments - Bob (Due: June 29)
- [ ] Create test plan - Carol (Due: July 1)

## Notes
Productive discussion. Team capacity at 85%.

## Next Meeting
July 1, 2026 at 2:00 PM"""
})

print("Agent output:")
print(result["messages"][-1].content)

# Verify file was created
print("\n" + "=" * 70)
print("VERIFICATION:")
print("=" * 70)

if os.path.exists("meeting_notes.md"):
    print("\n✅ File created successfully: meeting_notes.md")
    
    with open("meeting_notes.md", "r") as f:
        content = f.read()
    
    print(f"File size: {len(content)} bytes")
    print("\nFile contents:")
    print("-" * 70)
    print(content)
    print("-" * 70)
    print("\n✅ File verified and readable!")
else:
    print("\n❌ File was not created by agent")

print("=" * 70)

---

## Feature 2: Memory (AGENTS.md)

**Memory** persists across agent sessions using an `AGENTS.md` file convention (inspired by `CLAUDE.md`).

The file contains:
- **Preferences** — coding style, communication tone, defaults
- **Project context** — relevant facts about the domain
- **Known issues** — workarounds and constraints

Memory is automatically loaded at startup and always present in the system prompt, so the agent remembers preferences across sessions.

In [3]:
import os

# Create an AGENTS.md file in the workspace
agents_md = """# AGENTS.md

## Preferences
- Always write Python code with type hints and docstrings
- Use f-strings for formatting, not .format()
- Prefer async code where possible
- Keep functions under 50 lines

## Project Context
This is a data analysis project focused on financial time series.
Primary database: PostgreSQL 14
ORM: SQLAlchemy 2.0

## Known Issues
- The `transactions` table can be slow for large date ranges (>1 year)
  Use EXPLAIN ANALYZE before writing new queries
- Missing data in 2024-01 to 2024-03 — documented in JIRA-1234
"""

# Write the AGENTS.md to disk (agent will load it)
with open("AGENTS.md", "w") as f:
    f.write(agents_md)

print("✓ AGENTS.md created")
print("\nContent:")
print(agents_md)

✓ AGENTS.md created

Content:
# AGENTS.md

## Preferences
- Always write Python code with type hints and docstrings
- Use f-strings for formatting, not .format()
- Prefer async code where possible
- Keep functions under 50 lines

## Project Context
This is a data analysis project focused on financial time series.
Primary database: PostgreSQL 14
ORM: SQLAlchemy 2.0

## Known Issues
- The `transactions` table can be slow for large date ranges (>1 year)
  Use EXPLAIN ANALYZE before writing new queries
- Missing data in 2024-01 to 2024-03 — documented in JIRA-1234



In [4]:
# Create an agent that will load the AGENTS.md
agent = create_deep_agent(
    model=llm,
    tools=[],
    system_prompt="""You are a Python developer working on a financial data pipeline.
    Follow the preferences in AGENTS.md when writing code.
    Remember the known issues when designing queries.""",
)

# The agent will automatically load AGENTS.md from the current directory
result = agent.invoke({
    "messages": "Write a Python function to fetch transaction data from 2024, remembering the known issues and code preferences."
})

print("Agent's response (using preferences from AGENTS.md):")
print(result["messages"][-1].content[:600])
print("...")

Agent's response (using preferences from AGENTS.md):
It seems there are no existing files in the directory to reference for known issues or code preferences. I'll proceed with writing a Python function to fetch transaction data from 2024 based on general best practices.

```python
import requests

def fetch_transaction_data(year=2024):
    """
    Fetches transaction data for a given year.

    Parameters:
    - year: int, the year for which to fetch transaction data. Defaults to 2024.

    Returns:
    - dict: A dictionary containing transaction data.
    """
    # Base URL for the transaction data API
    base_url = "https://api.example.com/tr
...


---

## Feature 3: Skills (SKILL.md)

**Skills** implement "progressive disclosure" — specialized capabilities loaded on-demand instead of upfront.

A skill is a markdown file with YAML frontmatter that the agent can load when needed:

```yaml
---
name: data-analysis
description: Advanced statistical analysis with pandas and scipy
version: 1.0.0
---

# Data Analysis Skill

When this skill is active:
1. Always check data quality first (df.info(), missing values, outliers)
2. Use scipy for statistical tests
3. Save visualizations to ./output/
```

Benefits:
- **Reduces prompt bloat** — only load relevant skills
- **Reusable** — share skills across multiple agents
- **Versionable** — track skill versions independently

In [5]:
import os

# Create a skills directory
os.makedirs("skills", exist_ok=True)

# Create a data analysis skill
data_analysis_skill = """---
name: data-analysis
description: Advanced statistical analysis with pandas and matplotlib
version: 1.0.0
---

# Data Analysis Skill

When this skill is active, follow these patterns:

## Data Exploration
1. Always start with `df.info()` and `df.describe()`
2. Check for missing values: `df.isnull().sum()`
3. Look for outliers using IQR method

## Analysis Workflow
- Use pandas for data manipulation
- Use matplotlib for static plots
- Use scipy.stats for hypothesis testing
- Save all plots to `./output/` with descriptive names

## Output Format
Always save analysis results to `analysis_report.md` with:
- Summary statistics
- Key findings
- Recommendations
- References to generated plots
"""

# Write the skill
with open("skills/data-analysis.md", "w") as f:
    f.write(data_analysis_skill)

print("✓ Skill created: skills/data-analysis.md")
print("\nSkill content (first 300 chars):")
print(data_analysis_skill[:300] + "...")

✓ Skill created: skills/data-analysis.md

Skill content (first 300 chars):
---
name: data-analysis
description: Advanced statistical analysis with pandas and matplotlib
version: 1.0.0
---

# Data Analysis Skill

When this skill is active, follow these patterns:

## Data Exploration
1. Always start with `df.info()` and `df.describe()`
2. Check for missing values: `df.isnull...


---

## Feature 4: Subagents

**Subagents** let you delegate subtasks to specialized child agents running in isolated context windows.

There are two types:

### Inline Subagents (blocking)
- Run synchronously in the same process
- Main agent waits for completion
- Best for: fast, predictable tasks

### Async Subagents (background)
- Run asynchronously on a remote server
- Main agent continues while subagent works
- Best for: long-running tasks (minutes to hours)

Both types use the `task` tool to delegate work.

In [ ]:
from langchain_core.tools import tool
from tenacity import retry, stop_after_attempt, wait_exponential
import time

# Define a simple data source
SAMPLE_TOPICS = [
    "Machine Learning",
    "Web Development",
    "Cloud Architecture",
    "DevOps",
    "Data Engineering",
]

@tool
def get_topic_info(topic: str) -> str:
    """Get information about a technology topic."""
    info = {
        "Machine Learning": "ML uses algorithms to learn from data without explicit programming.",
        "Web Development": "Building web applications using HTML, CSS, JavaScript, and frameworks.",
        "Cloud Architecture": "Designing scalable systems using cloud providers like AWS, GCP, Azure.",
        "DevOps": "Automating deployment, monitoring, and infrastructure management.",
        "Data Engineering": "Building pipelines to process and store large-scale data.",
    }
    return info.get(topic, f"No info available for {topic}")

# Create a specialized researcher subagent (simple, focused agent)
researcher = create_deep_agent(
    model=llm,
    tools=[get_topic_info],
    system_prompt="""You are a tech researcher. 
    When asked about a topic, use get_topic_info to gather data,
    then write a comprehensive summary in 2-3 sentences.""",
)

# Retry wrapper for rate limit handling
@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=2, max=60),
    reraise=True
)
def researcher_invoke_with_retry(topic):
    """Invoke researcher with automatic retry on rate limits."""
    return researcher.invoke({"messages": f"Research and summarize: {topic}"})

# The main agent uses the researcher by calling .invoke() directly
# This is the inline subagent pattern
def coordinate_research(topics):
    results = []
    
    for i, topic in enumerate(topics):
        print(f"\n→ [{i+1}/{len(topics)}] Researching: {topic}")
        try:
            result = researcher_invoke_with_retry(topic)
            summary = result["messages"][-1].content
            results.append(f"**{topic}**: {summary}")
        except Exception as e:
            print(f"  ⚠️ Failed after retries: {type(e).__name__}")
            results.append(f"**{topic}**: (research failed)")
        
        # Add delay between topics to avoid rate limits
        if i < len(topics) - 1:
            time.sleep(1)
    
    return "\n\n".join(results)

# Use the subagent to research topics
research_summary = coordinate_research(SAMPLE_TOPICS[:3])

print("\nResearch Summary from Subagent:")
print("=" * 60)
print(research_summary)

---

## Feature 5: Streaming

**Streaming** returns responses token by token as the agent generates them, enabling real-time feedback and lower perceived latency.

DeepAgents supports streaming through `astream_events` (async generator) with different event types:

| Event Type | When |
|---|---|
| `on_llm_start` | LLM call begins |
| `on_llm_stream` | Token arrives from LLM |
| `on_tool_start` | Tool call begins |
| `on_tool_end` | Tool call completes |
| `on_chat_model_stream` | Chat model token arrives |

In [ ]:
import asyncio
import os

if not os.environ.get("OPENAI_API_KEY"):
    print("⚠️ Skipping streaming demo — requires OPENAI_API_KEY")
else:
    async def stream_agent_response():
        """Stream agent response token by token."""
        agent = create_deep_agent(
            model=llm,
            tools=[],
            system_prompt="You are a helpful assistant that explains concepts clearly and concisely.",
        )
        
        prompt = "Explain the difference between machine learning and deep learning in 3 sentences."
        print(f"Prompt: {prompt}\n")
        print("Streaming response:")
        print("-" * 50)
        
        # Stream events from the agent
        async for event in agent.astream_events(
            {"messages": prompt},
            version="v2",
        ):
            # Check for chat model streaming events
            if event["event"] == "on_chat_model_stream":
                token = event["data"]["chunk"].content
                print(token, end="", flush=True)
        
        print("\n" + "-" * 50)
        print("\n✓ Streaming complete")
    
    # Run the async streaming demo
    try:
        asyncio.run(stream_agent_response())
    except Exception as e:
        print(f"Error: {type(e).__name__}: {e}")
        print("Verify your OPENAI_API_KEY is valid")

---

## End-to-End Example: Research Report Generator

This example combines **all five features**:
1. **Filesystem** — saves the report to disk
2. **Memory** — remembers project preferences from AGENTS.md
3. **Skills** — loads the data-analysis skill
4. **Subagents** — delegates research to a child agent
5. **Streaming** — streams progress to the user

In [16]:
from tenacity import retry, stop_after_attempt, wait_exponential
import time

# Retry wrapper for rate limit handling
@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=2, max=60),
    reraise=True
)
def research_agent_invoke_with_retry(agent, prompt):
    """Invoke research agent with automatic retry on rate limits."""
    return agent.invoke({"messages": prompt})

# Create a research report generator using all five features
research_agent = create_deep_agent(
    model=llm,
    tools=[],
    system_prompt="""You are a research specialist.
    
    When researching topics:
    1. Use write_file to save research notes as you go
    2. Follow preferences from AGENTS.md
    3. When doing data analysis, activate the data-analysis skill
    4. Create a comprehensive report and save to report.md""",
)

# Ask the agent to research and generate a report
print("Starting research report generation...")
print("(This may take a minute due to rate limit handling)\n")

try:
    result = research_agent_invoke_with_retry(
        research_agent,
        """Research the top 3 Python data science libraries (NumPy, Pandas, Scikit-learn).
    For each:
    1. Write a brief description
    2. List key features
    3. Save findings to a file
    
    Then create a comparison report and save it as comparison.md"""
    )
    
    print("Research Report Generator Result:")
    print("-" * 60)
    print(result["messages"][-1].content[:500])
    print("...")
except Exception as e:
    print(f"Error: {type(e).__name__}")
    print(f"Message: {str(e)[:200]}")
    print("\n(Retried up to 5 times with exponential backoff)")

Starting research report generation...
(This may take a minute due to rate limit handling)

Research Report Generator Result:
------------------------------------------------------------
I have completed the research and saved the findings for each library in their respective files:

1. **NumPy**: [NumPy_Research.md](NumPy_Research.md)
2. **Pandas**: [Pandas_Research.md](Pandas_Research.md)
3. **Scikit-learn**: [Scikit_learn_Research.md](Scikit_learn_Research.md)

Additionally, a comparison report has been created and saved as [comparison.md](comparison.md). This report provides an overview and comparative insights into the roles and features of these three libraries.
...


---

## Summary

| Feature | Use Case | Key Tool |
|---|---|---|
| **Filesystem** | Organize work, persist outputs | `read_file`, `write_file`, `edit_file` |
| **Memory** | Cross-session context, preferences | `AGENTS.md` file |
| **Skills** | Specialized, reusable capabilities | `SKILL.md` files with YAML frontmatter |
| **Subagents** | Delegate specialized work | `task` tool or `async_subagents` |
| **Streaming** | Real-time feedback, lower latency | `astream_events` (async) |

**Next Steps:**
1. Combine these features into a production agent for your domain
2. Create domain-specific skills in `skills/*.md`
3. Define project memory in `AGENTS.md`
4. Build specialized subagents for heavy lifting
5. Stream results to users for real-time feedback